# Explore a trained checkpoint
Loads a checkpoint saved by `scripts/train.py` and plots. **Never trains**: restarting the kernel costs seconds.
All the logic lives in `src/pds_cd/`, so these plots match exactly what `evaluate.py` logs to W&B.

In [3]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config import ROOT
from datasets import build_dataset
from model import build_models
from metrics import collect_errors, best_f1_threshold, scores_at_threshold, separability
from utils import fixed_rng, get_device
from viz import error_histograms, pick_indices, predict_samples, visualize_predictions

device = get_device()
sorted((ROOT / "checkpoints").glob("*.pt"))  # available checkpoints

ModuleNotFoundError: No module named 'src.config'

In [ ]:
CKPT = ROOT / "checkpoints" / "REPLACE-ME.pt"   # <- pick one from the list above

ckpt = torch.load(CKPT, map_location=device)
cfg = ckpt["cfg"]                       # same config as training
enc, tr = build_models(cfg, device)
tr.load_state_dict(ckpt["transformer"])
{k: cfg[k] for k in ("dataset", "encoder", "crop", "augment", "seed", "epochs")}

In [ ]:
# errors on train (no augmentation) and val, same protocol as evaluate.py
def errors(split, augment=None):
    ds = build_dataset(cfg, split, augment=augment)
    dl = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=cfg["num_workers"])
    with fixed_rng(cfg["eval_seed"]):
        return ds, collect_errors(dl, enc, tr, device)

_, tr_err = errors("train", augment=False)
val_ds, val_err = errors("val")

thr = {m: best_f1_threshold(tr_err[m], tr_err["labels"])[0] for m in ("model", "raw")}
for m in ("model", "raw"):
    print(m, f"thr={thr[m]:.3f}", scores_at_threshold(val_err[m], val_err["labels"], thr[m]),
          separability(val_err[m], val_err["labels"]))

In [ ]:
error_histograms(val_err, thr, "val"); plt.show()

In [ ]:
idx = pick_indices(val_ds, n=10, seed=3)
im1, im2, labels, err_model, err_raw = predict_samples(enc, tr, val_ds, device, idx)
visualize_predictions(im1, im2, labels, err_model, threshold=thr["model"], indices=idx); plt.show()